<a href="https://colab.research.google.com/github/Ekliipce/Representation-and-Generative-Learning/blob/main/Reimpl%C3%A9mentation_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# RNN Reimplementation


We aim at writing a simple RNN that will be able to sum two binary strings.

Inputs will be successive bits that the network will have to add and the output is going to be the result bit.

In such a case, we obviously see why a RNN might work where a simple NN could not, because of the result of the previous addition needs to be kept in memory.

\begin{matrix}
& 1 & 0 & 1 & 1 \\
+ & 0 & 0 & 0 & 1 \\
= & 1 & 1 & 0 & 0 \\
\end{matrix}

In [1]:
import copy
import numpy as np
import torch
from tqdm import tqdm

In [2]:
def sigmoid(x):
    """
    Le calcul de la fonction sigmoid.
    """
    return 1 / (1 + np.exp(-x))


def d_sigmoid(x):
    """
    Le calcul de la déCopie de TPrivée de la fonction sigmoid.
    """
    sig = sigmoid(x)
    return sig * (1 - sig)


def d_sigmoid_easy(x):
    """
    Une variation de la dérivée de la sigmoid, pour simplifier certains calcul
    lors de la rétropropagation.
    """
    return x * (1 - x)
     


In [3]:
def generate_dataset(d):
    dataset = {}
    largest_number = pow(2, d)
    for i in range(largest_number):
        dataset[i] = np.array([(i >> j) & 1 for j in range(d-1, -1, -1)], dtype=np.uint8)
    return dataset


def get_sample(dataset):
    """
    Sélectionne deux nombres a et b dans l'ensemble de données et calcule la somme c.
    Si la somme est inférieure au nombre le plus grand du dictionnaire, renvoie :
    le tableau de bits de a, le tableau de bits de b, le tableau de bits de c
    -> dataset[a], dataset[b], dataset[c]
    """
    choices = list(dataset.keys())
    max = choices[-1]

    while(True):
      a = np.random.choice(choices)
      b = np.random.choice(choices)
      c = a + b
      if (c < max):
        bit_a = dataset[a]
        bit_b = dataset[b]
        bit_c = dataset[c]

        return np.array([bit_a, bit_b, bit_c])
      
def generate_samples(dataset, n_samples):
    """Génère n_samples exemples (a, b, c) où a + b = c."""
    samples = []
    choices = list(dataset.keys())
    max_val = choices[-1]
    
    while len(samples) < n_samples:
        a = int(np.random.choice(choices))
        b = int(np.random.choice(choices))
        c = a + b
        if c <= max_val:
            samples.append((dataset[a], dataset[b], dataset[c]))
    
    return samples

      
def train_test_split(samples, test_ratio: float):
    """Divise les samples en train et test sets."""
    np.random.shuffle(samples)
    split_idx = int(len(samples) * (1 - test_ratio))
    return samples[:split_idx], samples[split_idx:]

      
dataset = generate_dataset(10)
get_sample(dataset)

array([[1, 0, 1, 1, 0, 1, 1, 0, 1, 1],
       [0, 0, 1, 1, 0, 1, 1, 1, 0, 1],
       [1, 1, 1, 0, 1, 1, 1, 0, 0, 0]], dtype=uint8)

In [4]:
def bits_to_int(bits: np.ndarray) -> int:
    """Convertit un array de bits (big endian) en entier."""
    result = 0
    for bit in bits:
        result = (result << 1) | int(bit)
    return result


def int_to_bits(n: int, num_bits: int) -> np.ndarray:
    """Convertit un entier en array de bits (big endian)."""
    return np.array([(n >> (num_bits - 1 - i)) & 1 for i in range(num_bits)], dtype=np.uint8)


def format_addition(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> str:
    """Formate une addition binaire pour l'affichage."""
    return f"{a} + {b} = {c}\n" \
           f"{bits_to_int(a)} + {bits_to_int(b)} = {bits_to_int(c)}"

sample = get_sample(dataset)
a, b, c = sample
print(format_addition(a, b, c))


[0 1 1 0 0 0 0 0 0 0] + [1 0 0 1 1 0 1 1 0 1] = [1 1 1 1 1 0 1 1 0 1]
384 + 621 = 1005


## Forward
$$a_t = w_ix_t + w_hh_{t-1}$$

$$h_t = σ(a_t)$$

$$o_t = w_oh_t$$

$$ŷ = \sigma(o_t)$$



## Backward
With $L$, loss function as $\sum^N_{i=0}(ŷ - y)²$ : <br/><br/>

1. $$\frac{∂L}{\partial w_o} = \frac{∂L_t}{\partial o_t} \frac{∂o_t}{\partial w_o} = \frac{∂L_t}{\partial o_t} \frac{∂w_oh_t}{\partial w_o} = \frac{∂L_t}{\partial o_t} h_t$$
and $$\frac{∂L}{\partial o_t} = \frac{∂L_t}{\partial ŷ_t}\frac{∂ŷ_t}{\partial o_t} = \frac{∂(ŷ - y)²}{\partial ŷ_t}\frac{∂σ(o_t)}{\partial o_t} = 2(ŷ-y)*ŷ(1-ŷ)$$
so $$\frac{∂L}{\partial w_o} = 2(ŷ-y)*ŷ(1-ŷ) * h_t$$

___
<br/><br/>

2. $$\frac{\delta L}{\delta w_i} = \frac{\delta L}{\delta h_t}\frac{\delta h_t}{\delta a_t}\frac{\delta a_t}{\delta w_i} = \frac{\delta L}{\delta h_t}h_t(1-h_t)x_t$$

<br/><br/>
___

<br/><br/>

3. $$\frac{\delta L}{\delta w_h} = \frac{\delta L}{\delta h_t} . \frac{\delta h_t}{\delta a_t}.\frac{\delta a_t}{\delta w_h} = h_{t-1}h_t(1 - h_t)\frac{\delta L}{\delta h_t}$$

<br/><br/>

___

with

$$\frac{\delta L}{\delta h_t} = \frac{\delta L_t}{\delta h_t} + \frac{\delta L_{i > t}}{\delta h_t} = \frac{\delta L_t}{\delta o_t}.\frac{\delta o_t}{\delta h_t} + \frac{\delta L}{\delta h_{t+1}}.\frac{\delta h_{t+1}}{\delta a_{t+1}}.\frac{\delta a_{t+1}}{\delta h_t}$$

$$\frac{\delta L}{\delta h_t} = w_o\frac{\delta L_t}{\delta o_t} + w_h h_{t+1}(1 - h_{t+1})\frac{\delta L} {\delta h_{t+1}}$$


In [5]:
from dataclasses import dataclass

@dataclass
class Config:
    alpha: float = 0.1
    input_dimension: int = 2
    hidden_dimension: int = 16
    output_dimension: int = 1
    number_bits: int = 8
    iterations: int = 10000
    test_split: float = 0.2
    seed: int = 42


In [6]:
class RNNModel:
    def __init__(self, config: Config):
        self.config = config
        np.random.seed(config.seed)
        
        # Initialisation des poids (uniforme [-1, 1])
        self.wi = 2 * np.random.random((config.hidden_dimension, config.input_dimension)) - 1
        self.wh = 2 * np.random.random((config.hidden_dimension, config.hidden_dimension)) - 1
        self.wo = 2 * np.random.random((config.output_dimension, config.hidden_dimension)) - 1
    
    def forward(
        self, 
        a: np.ndarray, 
        b: np.ndarray
    ):
        """
        Forward pass sur une paire (a, b).
        
        Returns:
            predictions: array des bits prédits
            ht_values: liste des états cachés
            o_deltas: liste des gradients de sortie (pour le backward)
            inputs: liste des inputs à chaque pas de temps
        """
        predictions = np.zeros(self.config.number_bits, dtype=np.uint8)
        ht_values = [np.zeros((self.config.hidden_dimension, 1))]
        inputs = []
        outputs = []  
        
        # Du LSB au MSB
        for pos in range(self.config.number_bits - 1, -1, -1):
            x = np.array([[a[pos]], [b[pos]]])
            inputs.append(x)
            
            # RNN cell
            a_t = self.wi @ x + self.wh @ ht_values[-1]
            h_t = sigmoid(a_t)
            ht_values.append(h_t)
            
            # Output
            o_t = self.wo @ h_t
            y_pred = sigmoid(o_t)
            outputs.append(y_pred)
            
            predictions[pos] = np.around(y_pred[0, 0])
        
        return predictions, ht_values, outputs, inputs
    
    def compute_loss(self, outputs, c):
        """
        Calcule la loss MSE et les o_deltas.
        
        Returns:
            total_error: somme des erreurs quadratiques
            o_deltas: gradients pour chaque pas de temps
        """
        total_error = 0.0
        o_deltas = []
        
        for t, y_pred in enumerate(outputs):
            pos = self.config.number_bits - 1 - t  # Mapping t -> pos
            y = np.array([[c[pos]]])
            
            error = (y_pred - y) ** 2
            total_error += error[0, 0]
            
            o_delta = (y_pred - y) * d_sigmoid_easy(y_pred)
            o_deltas.append(o_delta)
        
        return total_error, o_deltas
    
    def backward(self, inputs, ht_values, o_deltas):
        """
        Backward pass (BPTT).
        
        Returns:
            wi_grad, wh_grad, wo_grad: gradients des poids
        """
        wi_grad = np.zeros_like(self.wi)
        wh_grad = np.zeros_like(self.wh)
        wo_grad = np.zeros_like(self.wo)
        future_h_delta = np.zeros((self.config.hidden_dimension, 1))
        
        for t in reversed(range(self.config.number_bits)):
            x = inputs[t]
            h_t = ht_values[t + 1]
            h_t_prev = ht_values[t]
            o_delta = o_deltas[t]
            
            # Gradient de h_t
            h_delta = (self.wo.T @ o_delta) + (self.wh.T @ future_h_delta)
            
            # Gradient de a_t
            a_t_delta = h_delta * d_sigmoid_easy(h_t)
            
            # Accumulation
            wo_grad += o_delta @ h_t.T
            wh_grad += a_t_delta @ h_t_prev.T
            wi_grad += a_t_delta @ x.T
            
            future_h_delta = a_t_delta
        
        return wi_grad, wh_grad, wo_grad
    
    def update_weights(
        self, 
        wi_grad: np.ndarray, 
        wh_grad: np.ndarray, 
        wo_grad: np.ndarray
    ):
        """Met à jour les poids avec SGD."""
        self.wi -= self.config.alpha * wi_grad
        self.wh -= self.config.alpha * wh_grad
        self.wo -= self.config.alpha * wo_grad
    
    def predict(self, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Prédit c = a + b (inference only)."""
        predictions, _, _, _ = self.forward(a, b)
        return predictions



In [7]:
def train_epoch(model: RNNModel, train_samples):
    """Entraîne le modèle sur une époque complète."""
    total_loss = 0.0
    
    for a, b, c in train_samples:
        # Forward
        predictions, ht_values, outputs, inputs = model.forward(a, b)
        
        # Loss
        loss, o_deltas = model.compute_loss(outputs, c)
        total_loss += loss
        
        # Backward
        wi_grad, wh_grad, wo_grad = model.backward(inputs, ht_values, o_deltas)
        
        # Update
        model.update_weights(wi_grad, wh_grad, wo_grad)
    
    return total_loss / len(train_samples)

def evaluate(model: RNNModel, test_samples):
    """
    Évalue le modèle sur le test set.
    
    Returns:
        accuracy: pourcentage de prédictions parfaites
        avg_loss: loss moyenne
    """
    correct = 0
    total_loss = 0.0
    
    for a, b, c in test_samples:
        predictions, _, outputs, _ = model.forward(a, b)
        loss, _ = model.compute_loss(outputs, c)
        total_loss += loss
        
        if np.array_equal(predictions, c):
            correct += 1
    
    accuracy = correct / len(test_samples)
    avg_loss = total_loss / len(test_samples)
    
    return accuracy, avg_loss


def train(config: Config, verbose: bool = True) -> RNNModel:
    """Pipeline d'entraînement complet."""
    # Dataset
    dataset = generate_dataset(config.number_bits)
    samples = generate_samples(dataset, n_samples=config.iterations)
    train_samples, test_samples = train_test_split(samples, config.test_split)
    
    if verbose:
        print(f"Train samples: {len(train_samples)}, Test samples: {len(test_samples)}")
    
    # Model
    model = RNNModel(config)
    
    # Training
    n_epochs = 10
    samples_per_epoch = len(train_samples) // n_epochs
    
    for epoch in range(n_epochs):
        start_idx = epoch * samples_per_epoch
        end_idx = start_idx + samples_per_epoch
        epoch_samples = train_samples[start_idx:end_idx]
        
        avg_loss = train_epoch(model, epoch_samples)
        
        if verbose and epoch % 2 == 0:
            acc, test_loss = evaluate(model, test_samples[:100])
            print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_loss:.4f} | Test Acc: {acc*100:.1f}%")
            
            # Exemple
            a, b, c = test_samples[0]
            pred = model.predict(a, b)
            print(f"  Example: {format_addition(a, b, c)}")
            print(f"  Predicted: {format_addition(a, b, pred)} {'✓' if np.array_equal(pred, c) else '✗'}")
    
    # Final evaluation
    if verbose:
        acc, test_loss = evaluate(model, test_samples)
        print(f"\nFinal Test Accuracy: {acc*100:.1f}%")
    
    return model

In [8]:
config = Config(
    alpha=0.1,
    hidden_dimension=16,
    number_bits=8,
    iterations=10000,
    test_split=0.2
)
    
model = train(config)



Train samples: 8000, Test samples: 2000
Epoch 1/10 | Train Loss: 1.9858 | Test Acc: 0.0%
  Example: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 0 1 1 1 1 1]
138 + 21 = 159
  Predicted: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 1 1 1 1 1 1 0]
138 + 21 = 254 ✗
Epoch 3/10 | Train Loss: 1.8371 | Test Acc: 4.0%
  Example: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 0 1 1 1 1 1]
138 + 21 = 159
  Predicted: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 1 1 1 1 1 1 1]
138 + 21 = 255 ✗
Epoch 5/10 | Train Loss: 1.3604 | Test Acc: 40.0%
  Example: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 0 1 1 1 1 1]
138 + 21 = 159
  Predicted: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 0 1 1 1 1 1]
138 + 21 = 159 ✓
Epoch 7/10 | Train Loss: 0.7203 | Test Acc: 64.0%
  Example: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 0 1 1 1 1 1]
138 + 21 = 159
  Predicted: [1 0 0 0 1 0 1 0] + [0 0 0 1 0 1 0 1] = [1 0 1 1 1 1 0 1]
138 + 21 = 189 ✗
Epoch 9/10 | Train Loss: 0.2487 | Test Acc: 99.0%
  Example: [1 0 0 0 1 0 

In [9]:
# Test manuel
print("\n--- Tests manuels ---")
dataset = generate_dataset(config.number_bits)
for a_int, b_int in [(5, 3), (42, 17), (100, 27)]:
    a, b = dataset[a_int], dataset[b_int]
    c_true = int_to_bits(a_int + b_int, config.number_bits)
    c_pred = model.predict(a, b)
    
    status = "✓" if np.array_equal(c_pred, c_true) else "✗"
    print(f"{a_int} + {b_int} = {bits_to_int(c_pred)} (expected {a_int + b_int}) {status}")


--- Tests manuels ---
5 + 3 = 8 (expected 8) ✓
42 + 17 = 59 (expected 59) ✓
100 + 27 = 127 (expected 127) ✓
